# 00 · Preparación del entorno
Crea el catálogo, los esquemas `bronze / silver / gold` y el volumen donde se dejan los ficheros de cada lote.

**Se ejecuta una sola vez.** Es idempotente (`IF NOT EXISTS`).

Cómputo: *serverless* (es el único disponible en Free Edition y arranca en segundos para cargas pequeñas).
Todas las tablas son *managed* (Free Edition no permite ubicaciones de almacenamiento propias, así que no hay tablas *external*).

In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))  # raíz del repo -> `import src...`
from src.config import ESQUEMAS, VOLUMEN, CATALOGO_POR_DEFECTO

dbutils.widgets.text("catalogo", CATALOGO_POR_DEFECTO)
catalogo = dbutils.widgets.get("catalogo")

## 1. Catálogo
Si tu workspace **no permite crear catálogos**, el `CREATE CATALOG` fallará: en ese caso usa el catálogo del workspace
(el que ves en *Catalog*, con el nombre de tu workspace) y pon ese nombre en el parámetro `catalogo` de **todos** los notebooks y del job.

In [0]:
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo} COMMENT 'Proyecto DP-750: logística DataCo (incremental + SCD2)'")
    print(f"Catálogo listo: {catalogo}")
except Exception as e:
    print("No se pudo crear el catálogo:", str(e)[:300])
    print("-> Usa el catálogo del workspace y repite este notebook con ese nombre en el parámetro 'catalogo'.")
    raise

## 2. Esquemas (capas Medallion) y volumen de entrada

In [0]:
comentarios = {
    "bronze": "Datos tal cual llegan (texto), con metadatos de carga. Sin columnas sensibles.",
    "silver": "Datos validados, tipados y deduplicados + rechazos, métricas y log de cargas.",
    "gold": "Modelo dimensional (dim_cliente SCD2, dim_producto, dim_ubicacion, fact_envios) y data marts.",
}
for esquema in ESQUEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema} COMMENT '{comentarios[esquema]}'")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.bronze.{VOLUMEN} COMMENT 'Ficheros CSV de entrada, una carpeta por lote'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.bronze.raw COMMENT 'CSV original de Kaggle (sin dividir en lotes)'")

display(spark.sql(f"SHOW SCHEMAS IN {catalogo}"))

## 3. Siguiente paso
Sube el CSV de Kaggle al volumen `raw` (*Catalog → bronze → raw → Upload to this volume*) y ejecuta
`00b_generar_lotes`, que crea las carpetas de lotes en `landing`. Detalle en `paso_a_paso.md`.

In [0]:
print(f"Ruta de entrada: /Volumes/{catalogo}/bronze/{VOLUMEN}/<lote>/")